# Implementacao Completa: Classificacao Binaria (2 Classes)

Este notebook segue o template de classificacao e aplica um fluxo completo para o dataset binario do projeto.

Objetivos deste notebook:
- Carregar o arquivo final `../data/data_set_final.csv` da versao binaria.
- Realizar analise exploratoria (EDA) com graficos para entender distribuicoes e relacoes.
- Preparar os dados sem alterar os valores finais do dataset.
- Treinar um modelo binario com 2 classes e avaliar acuracia e metricas por classe.

Mapeamento da classe alvo (`CLASSI_FIN`):
- 0 = OUTRAS_DOENCAS
- 1 = DENGUE


In [ ]:
import warnings
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
from tensorflow.keras.callbacks import EarlyStopping

# Configuracoes visuais e de logs para facilitar analise.
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)
gpu = tf.config.experimental.list_physical_devices('GPU')

print("TensorFlow version:", tf.__version__)
print("Built with CUDA:", tf.test.is_built_with_cuda())
print("Visible GPUs:", gpu)

## STEP 1: Load Data (Dataset Final)

In [ ]:
# Caminho oficial do dataset final gerado pelo script de codificacao.
DATA_PATH = "../data/data_set_final.csv"
TARGET_COL = "CLASSI_FIN"
CLASS_NAME_MAP = {0: "OUTRAS_DOENCAS", 1: "DENGUE"}

df = pd.read_csv(DATA_PATH)

# Garantimos que a coluna alvo seja numerica para treino binario.
df[TARGET_COL] = pd.to_numeric(df[TARGET_COL], errors="raise").astype(int)

print("Dataset carregado com sucesso:", DATA_PATH)
print("Shape:", df.shape)
print("Colunas:", len(df.columns))
df.head()

## STEP 2: Data Inspection and Quality Checks

In [ ]:
# Inspecao estrutural para validar se os dados finais estao prontos para modelagem.
print("Tipos de dados:")
print(df.dtypes)

print("\nValores ausentes por coluna:")
print(df.isnull().sum())

print("\nDistribuicao da classe alvo (codigo):")
print(df[TARGET_COL].value_counts().sort_index())

# Estatisticas descritivas das variaveis numericas.
df.describe().T.head(10)

## STEP 3: Visualize Target Distribution

In [ ]:
# Grafico da distribuicao das classes para confirmar o balanceamento do dataset.
class_counts = df[TARGET_COL].value_counts().sort_index()
labels = [CLASS_NAME_MAP.get(i, str(i)) for i in class_counts.index]

ax = sns.barplot(x=labels, y=class_counts.values, palette="viridis")
ax.set_title("Distribuicao de Classes (CLASSI_FIN)")
ax.set_xlabel("Classe")
ax.set_ylabel("Quantidade de Registros")

for idx, value in enumerate(class_counts.values):
    ax.text(idx, value + 20, str(value), ha="center", fontsize=10)

plt.tight_layout()
plt.show()

## STEP 4: Correlation Analysis

In [ ]:
# Mapa de calor para observar relacoes lineares entre atributos numericos.
plt.figure(figsize=(14, 10))
correlation = df.corr(numeric_only=True)
sns.heatmap(correlation, cmap="coolwarm", center=0)
plt.title("Matriz de Correlacao - Dataset Binario")
plt.tight_layout()
plt.show()

## STEP 5: Prepare Features and Target

In [ ]:
# Separamos atributos (X) e alvo (y) sem alterar valores do dataset final.
X = df.drop(columns=[TARGET_COL]).values
y = df[TARGET_COL].astype(int).values

classes = np.unique(y)
print("Detected classes:", classes)
if len(classes) != 2:
    raise ValueError(f"Expected exactly 2 classes, but found {len(classes)}: {classes}")
if set(classes.tolist()) != {0, 1}:
    raise ValueError("As classes esperadas para CLASSI_FIN sao exatamente 0 e 1.")

print("X shape:", X.shape)
print("y shape:", y.shape)

## STEP 6: Train/Test Split and Feature Scaling

In [ ]:
# Split estratificado para manter proporcoes de classe no treino e teste.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

# Verificacao explicita das proporcoes por classe para garantir imparcialidade.
full_ratio = pd.Series(y).value_counts(normalize=True).sort_index()
train_ratio = pd.Series(y_train).value_counts(normalize=True).sort_index()
test_ratio = pd.Series(y_test).value_counts(normalize=True).sort_index()

counts_df = pd.DataFrame(
    {
        "full_count": pd.Series(y).value_counts().sort_index(),
        "train_count": pd.Series(y_train).value_counts().sort_index(),
        "test_count": pd.Series(y_test).value_counts().sort_index(),
        "full_ratio": full_ratio,
        "train_ratio": train_ratio,
        "test_ratio": test_ratio,
    }
)

print("Distribuicao por classe apos split:")
display(counts_df)

# Tolerancia pequena para arredondamentos de contagem.
if not np.allclose(train_ratio.values, full_ratio.values, atol=1e-3):
    raise ValueError("Proporcao das classes no treino difere da base completa.")
if not np.allclose(test_ratio.values, full_ratio.values, atol=1e-3):
    raise ValueError("Proporcao das classes no teste difere da base completa.")

# O scaler e ajustado apenas no treino para evitar leakage.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("X_train shape:", X_train_scaled.shape)
print("y_train shape:", y_train.shape)
print("X_test shape:", X_test_scaled.shape)
print("y_test shape:", y_test.shape)

## STEP 7: Build and Compile Binary Model

In [ ]:
# Arquitetura base para classificacao binaria.
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(X_train_scaled.shape[1],)),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dropout(0.25),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dropout(0.25),
    tf.keras.layers.Dense(1, activation="sigmoid"),
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

model.summary()

## STEP 8: Train Model

In [ ]:
# Early stopping reduz risco de overfitting em treinos longos.
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=15,
    restore_best_weights=True,
)

history = model.fit(
    X_train_scaled,
    y_train,
    epochs=120,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stopping],
    verbose=1,
)

## STEP 9: Evaluate Accuracy and Classification Metrics

In [ ]:
test_loss, test_acc = model.evaluate(X_test_scaled, y_test, verbose=0)
print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_acc:.4f}")

# Predicao final com limiar de 0.5 sobre probabilidades da sigmoid.
probs = model.predict(X_test_scaled, verbose=0).ravel()
y_pred = (probs >= 0.5).astype(int)

print("\nAccuracy (sklearn):", f"{accuracy_score(y_test, y_pred):.4f}")

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(7, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=[CLASS_NAME_MAP[i] for i in sorted(CLASS_NAME_MAP)],
    yticklabels=[CLASS_NAME_MAP[i] for i in sorted(CLASS_NAME_MAP)],
)
plt.title("Confusion Matrix - Test Set")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.tight_layout()
plt.show()

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred,
        target_names=[CLASS_NAME_MAP[i] for i in sorted(CLASS_NAME_MAP)],
        digits=4,
    )
)

## STEP 10: Training Curves (Loss and Accuracy)

In [ ]:
# Curvas de treino e validacao para analisar convergencia e overfitting.
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history["loss"], label="Train Loss")
axes[0].plot(history.history["val_loss"], label="Val Loss")
axes[0].set_title("Loss por Epoca")
axes[0].set_xlabel("Epoca")
axes[0].set_ylabel("Loss")
axes[0].legend()

axes[1].plot(history.history["accuracy"], label="Train Accuracy")
axes[1].plot(history.history["val_accuracy"], label="Val Accuracy")
axes[1].set_title("Accuracy por Epoca")
axes[1].set_xlabel("Epoca")
axes[1].set_ylabel("Accuracy")
axes[1].legend()

plt.tight_layout()
plt.show()